# Generating document id on page index

In [1]:
# Imports
import pageindex.utils as utils
import sys
from pathlib import Path

# Define root directory for notebook context
root_dir = Path.cwd().parent  # Va desde pipelines/ a la raiz del proyecto
sys.path.insert(0, str(root_dir))  # Agregar al path para importaciones

from core.config import settings
from core.client import gemini_client, pi_client
import json

# SUBMIT DOC TO PAGE INDEX

In [2]:

pdf_path = root_dir / "data" / "raw_files" / "quechua_grammar.pdf"
doc_id = pi_client.submit_document(str(pdf_path))["doc_id"]
print('Document Submitted:', doc_id)

Document Submitted: pi-cml4je2es03tz0fqzdngcb4yf


# Creating the structure

In [ ]:
doc_id = "pi-cmkxfip4n02f80hpesma22pm9"
if pi_client.is_retrieval_ready(doc_id):
    tree = pi_client.get_tree(doc_id, node_summary=True)['result']
    #utils.print_tree(tree)
    
    # guardar en un archivo JSON
    output_path = root_dir / "data" / "quechua_tree.json"
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(tree, f, indent=2, ensure_ascii=False)
    print(f'\nTree saved to: {output_path}')
else:
    print("Processing document, please try again later...")

Processing document, please try again later...


In [ ]:
utils.print_tree(tree)

# Reasoning-Based Retrieval with Tree Search

In [8]:
# Cargar el árbol desde el JSON guardado
import json

with open(root_dir / "data" / "quechua_tree.json", 'r', encoding='utf-8') as f:
    tree = json.load(f)

print(f"Tree loaded with {len(tree)} nodes")

Tree loaded with 2 nodes


## Step 1: Tree Search - Identificar nodos relevantes

In [9]:
# Definir la consulta sobre gramática quechua
query = "¿Cuáles son las reglas de formación de plural en quechua?"

# Eliminar el campo 'text' del árbol para el prompt (solo usar títulos y resúmenes)
tree_without_text = utils.remove_fields(tree.copy(), fields=['text'])

search_prompt = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{json.dumps(tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2", ..., "node_id_n"]
}}
Directly return the final JSON structure. Do not output anything else.
"""

# Usar Gemini para búsqueda en el árbol (await porque es async)
tree_search_result = await gemini_client.generate_text(search_prompt)
print("Tree Search Result:")
print(tree_search_result)

Tree Search Result:
{
    "thinking": "The question asks for the rules of plural formation in Quechua. In Quechua morphology, pluralization applies to nouns (nominal suffixes), pronouns, and verbs (verbal suffixes). \n\n1.  **Node 0015** ('Quechua Word Structure and Classification') explicitly states in its summary that it details 'Dependent nominal suffixes... including flexive suffixes for... number (singular and plural forms...)'. This covers the nominal pluralization rules (e.g., the suffix *-kuna*).\n2.  **Node 0023** ('Flexión de número') focuses on 'verb conjugations... covering number (pluralization)'. This covers the rules for verbal pluralization (e.g., suffixes like *-chik*, *-ku*).\n3.  **Node 0028** ('Morphology: Pronouns and Nouns in Quechua') mentions 'Special attention is given to the nuances of plural forms for personal and possessive pronouns' and 'examples of nominal flexional dependent suffixes related to Number'. This covers the specific plural forms for pronouns a

## Step 2: Mostrar nodos recuperados y razonamiento
### Opcional

In [11]:
# Crear mapa de nodos para acceso rápido
node_map = utils.create_node_mapping(tree)

# Parsear resultado JSON
tree_search_result_json = json.loads(tree_search_result)

print('Reasoning Process:')
utils.print_wrapped(tree_search_result_json['thinking'])

print('\nRetrieved Nodes:')
for node_id in tree_search_result_json["node_list"]:
    node = node_map[node_id]
    print(f"Node ID: {node['node_id']}\t Page: {node['page_index']}\t Title: {node['title']}")

Reasoning Process:
The question asks for the rules of plural formation in Quechua. In Quechua morphology, pluralization
applies to nouns (nominal suffixes), pronouns, and verbs (verbal suffixes).

1.  **Node 0015** ('Quechua Word Structure and Classification') explicitly states in its summary
that it details 'Dependent nominal suffixes... including flexive suffixes for... number (singular
and plural forms...)'. This covers the nominal pluralization rules (e.g., the suffix *-kuna*).
2.  **Node 0023** ('Flexión de número') focuses on 'verb conjugations... covering number
(pluralization)'. This covers the rules for verbal pluralization (e.g., suffixes like *-chik*,
*-ku*).
3.  **Node 0028** ('Morphology: Pronouns and Nouns in Quechua') mentions 'Special attention is given
to the nuances of plural forms for personal and possessive pronouns' and 'examples of nominal
flexional dependent suffixes related to Number'. This covers the specific plural forms for pronouns
and reinforces the nominal

## Step 3: Extraer contexto relevante de los nodos

In [15]:
# Extraer el texto completo de los nodos relevantes
node_list = tree_search_result_json["node_list"]
relevant_content = "\n\n".join(node_map[node_id]["text"] for node_id in node_list)

#print('Retrieved Context:\n')
#utils.print_wrapped(relevant_content[:1500] + '...' if len(relevant_content) > 1500 else relevant_content)

## Step 4: Generar respuesta basada en el contexto

In [16]:
answer_prompt = f"""
Answer the question based on the context:

Question: {query}
Context: {relevant_content}

Provide a clear, concise answer based only on the context provided.
"""

print('Generated Answer:\n')
answer = await gemini_client.generate_text(answer_prompt)
utils.print_wrapped(answer)

# Guardar la respuesta en un archivo de texto
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = root_dir / "data" / f"quechua_answer_{timestamp}.txt"

with open(output_file, 'w', encoding='utf-8') as f:
    f.write(f"Query: {query}\n")
    f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("="*80 + "\n\n")
    f.write("Answer:\n")
    f.write(answer)
    f.write("\n\n" + "="*80 + "\n")
    f.write(f"Retrieved Nodes ({len(node_list)}):\n")
    for node_id in node_list:
        node = node_map[node_id]
        f.write(f"- Node {node['node_id']} (Page {node['page_index']}): {node['title']}\n")

print(f'\nAnswer saved to: {output_file}')

Generated Answer:

Basado en el documento proporcionado, las reglas de formación del plural (flexión de número) en
quechua son las siguientes:

1.  **Para los nombres (sustantivos):**
    *   **Singular:** No lleva ningún sufijo específico.
    *   **Plural:** Se añade el sufijo pluralizador **–kuna**.
    *   *Ejemplos:* Chukcha–kuna (cabellos), Qallu–kuna (lenguas), Kiru–kuna (dientes).

2.  **Para pluralizar personas (marcas de persona en posesivos o verbos):**
    Se utilizan sufijos específicos para indicar que varias personas realizan la acción o poseen
algo:
    *   **–nchik**: Primera persona inclusiva (yo y tú/ustedes). Ejemplo: *Wawqi–nchik* (nuestro
hermano).
    *   **–yku**: Primera persona exclusiva (yo y otros, pero no tú). Ejemplo: *Wawqi–yku* (nuestro
hermano).
    *   **–chik**: Segunda persona (tú -> ustedes). Ejemplo: *Wawqi–yki–chik* (tus hermanos [de
ustedes]).
    *   **–ku**: Tercera persona (él/ella -> ellos/ellas). Ejemplo: *Wawqi–n–ku* (sus hermanos).

Además

# Procesando nuevo documento: Gramar

In [3]:
# Subir el nuevo documento gramar a Page Index
gramar_pdf_path = root_dir / "data" / "raw_files" / "gramar.pdf"
gramar_doc_id = pi_client.submit_document(str(gramar_pdf_path))["doc_id"]
print('Gramar Document Submitted:', gramar_doc_id)

Gramar Document Submitted: pi-cmm8f9zqw0bak0mo9n4png4gr


## Generar estructura del documento Gramar

In [4]:
# Reemplaza con el doc_id generado en la celda anterior
gramar_doc_id = 'pi-cmm8f9zqw0bak0mo9n4png4gr'
#gramar_doc_id = "pi-cml4jndqk0cbj09qzqxxg8vba"

if pi_client.is_retrieval_ready(gramar_doc_id):
    gramar_tree = pi_client.get_tree(gramar_doc_id, node_summary=True)['result']
    #utils.print_tree(gramar_tree)
    
    # Guardar en un archivo JSON
    gramar_output_path = root_dir / "data" / "gramar_tree.json"
    with open(gramar_output_path, 'w', encoding='utf-8') as f:
        json.dump(gramar_tree, f, indent=2, ensure_ascii=False)
    print(f'\nGramar tree saved to: {gramar_output_path}')
else:
    print("Processing document, please try again later...")


Gramar tree saved to: /home/totora/Documents/QUECHUA TRANSLATOR/page-index-test/data/gramar_tree.json


# Traducción de frases en Quechua usando ambos documentos

In [5]:
# Cargar ambos árboles JSON
with open(root_dir / "data" / "quechua_tree.json", 'r', encoding='utf-8') as f:
    quechua_tree = json.load(f)

with open(root_dir / "data" / "gramar_tree.json", 'r', encoding='utf-8') as f:
    gramar_tree = json.load(f)

print(f"Quechua tree loaded: {len(quechua_tree)} nodes")
print(f"Gramar tree loaded: {len(gramar_tree)} nodes")

Quechua tree loaded: 2 nodes
Gramar tree loaded: 64 nodes


## Step 1: Tree Search en ambos documentos

In [8]:
# Frases en quechua para traducir
translation_query = """
Traduce las siguientes 5 frases del quechua al español:
1. Ñuqa hatun wasipi tiyani
2. Qam imaynataq kasanki
3. Paykunaqa chakrapi llank'anku
4. Inti rikcharimuchkan
5. Mama wayk'usqanta mikhuni
"""

# Preparar ambos árboles sin el campo 'text'
quechua_tree_without_text = utils.remove_fields(quechua_tree.copy(), fields=['text'])
gramar_tree_without_text = utils.remove_fields(gramar_tree.copy(), fields=['text'])

# Prompt combinado para buscar en ambos documentos
combined_search_prompt = f"""
You are given a question and two tree structures from different documents about Quechua language.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes from BOTH documents that are likely to contain the answer to the question.

Question: {translation_query}

Document 1 - Quechua Grammar tree structure:
{json.dumps(quechua_tree_without_text, indent=2)}

Document 2 - Gramar tree structure:
{json.dumps(gramar_tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes from both documents are relevant to the question>",
    "quechua_nodes": ["node_id_1", "node_id_2", ...],
    "gramar_nodes": ["node_id_1", "node_id_2", ...]
}}
Directly return the final JSON structure. Do not output anything else.
"""

# Buscar en ambos árboles
translation_search_result = await gemini_client.generate_text(combined_search_prompt)
print("Translation Search Result:")
print(translation_search_result)

Translation Search Result:
{
    "thinking": "To translate the given Quechua phrases, I need to identify nodes that explain the specific vocabulary (pronouns, nouns, verbs, adjectives) and grammatical structures (suffixes for case, tense, aspect, subordination) used in each sentence.\n\n1.  **'Ñuqa hatun wasipi tiyani'** (I live in the big house):\n    *   Requires: Pronouns ('Ñuqa'), Adjectives ('hatun'), Locative case ('-pi'), Verb 'tiyay' conjugation (1st person '-ni').\n    *   **Doc 1:** Node 0016 (Case suffixes), Node 0021 (Verb person flexion), Node 0028 (Pronouns), Node 0030 (Adjectives).\n    *   **Doc 2:** Node 0037 (Present conjugation intro), Node 0038 (Present simple, specifically 'tiyay'), Node 0044 (Pronouns), Node 0051 (Case suffixes including '-pi').\n\n2.  **'Qam imaynataq kasanki'** (How are you?):\n    *   Requires: Pronouns ('Qam'), Interrogatives ('imayna'), Suffix '-taq', Verb 'kay' (to be) conjugation.\n    *   **Doc 1:** Node 0026 (Validating/Connective suffixe

## Step 2: Mostrar nodos recuperados de ambos documentos

In [9]:
# Crear mapas de nodos para ambos documentos
quechua_node_map = utils.create_node_mapping(quechua_tree)
gramar_node_map = utils.create_node_mapping(gramar_tree)

# Parsear resultado JSON
translation_result_json = json.loads(translation_search_result)

print('Reasoning Process:')
utils.print_wrapped(translation_result_json['thinking'])

print('\n--- Retrieved Nodes from Quechua Grammar Document ---')
for node_id in translation_result_json.get("quechua_nodes", []):
    node = quechua_node_map[node_id]
    print(f"Node ID: {node['node_id']}\t Page: {node['page_index']}\t Title: {node['title']}")

print('\n--- Retrieved Nodes from Gramar Document ---')
for node_id in translation_result_json.get("gramar_nodes", []):
    node = gramar_node_map[node_id]
    print(f"Node ID: {node['node_id']}\t Page: {node['page_index']}\t Title: {node['title']}")

Reasoning Process:
To translate the given Quechua phrases, I need to identify nodes that explain the specific
vocabulary (pronouns, nouns, verbs, adjectives) and grammatical structures (suffixes for case,
tense, aspect, subordination) used in each sentence.

1.  **'Ñuqa hatun wasipi tiyani'** (I live in the big house):
    *   Requires: Pronouns ('Ñuqa'), Adjectives ('hatun'), Locative case ('-pi'), Verb 'tiyay'
conjugation (1st person '-ni').
    *   **Doc 1:** Node 0016 (Case suffixes), Node 0021 (Verb person flexion), Node 0028 (Pronouns),
Node 0030 (Adjectives).
    *   **Doc 2:** Node 0037 (Present conjugation intro), Node 0038 (Present simple, specifically
'tiyay'), Node 0044 (Pronouns), Node 0051 (Case suffixes including '-pi').

2.  **'Qam imaynataq kasanki'** (How are you?):
    *   Requires: Pronouns ('Qam'), Interrogatives ('imayna'), Suffix '-taq', Verb 'kay' (to be)
conjugation.
    *   **Doc 1:** Node 0026 (Validating/Connective suffixes like '-taq'), Node 0028 (Pronouns)

## Step 3: Extraer y combinar contexto de ambos documentos

In [11]:
# Extraer contexto de ambos documentos
quechua_nodes_list = translation_result_json.get("quechua_nodes", [])
gramar_nodes_list = translation_result_json.get("gramar_nodes", [])

quechua_content = "\n\n".join(quechua_node_map[node_id]["text"] for node_id in quechua_nodes_list)
gramar_content = "\n\n".join(gramar_node_map[node_id]["text"] for node_id in gramar_nodes_list)

# Combinar el contenido relevante
combined_content = f"""
=== Context from Quechua Grammar Document ===
{quechua_content}

=== Context from Gramar Document ===
{gramar_content}
"""

print('Combined Retrieved Context (preview):')
utils.print_wrapped(combined_content[:2000] + '...' if len(combined_content) > 2000 else combined_content)

Combined Retrieved Context (preview):

=== Context from Quechua Grammar Document ===
Wawqi –yku ‘nuestro hermano’
(exclusivo)

Wawqi –yki –chik ‘tus hermanos’

Wawqi –n –ku ‘sus hermanos’

c) El de caso:

Se llama así a las diferentes funciones que el nombre o el pronombre desempeñan en la oración.

En quechua el caso se indica añadiendo un sufijo que marca las funciones del sustantivo. Encontramos
un promedio de doce marcas de casos. Un solo sistema de casos para todo un sistema nominal. Que son:

&gt; Caso Nominativo:
En el quechua no tiene marca.

Ejemplos:
Manka ‘olla’
Yanta ‘leña’
Pirqa ‘pared’

&gt; Caso Acusativo:
Designa el objeto directo con los verbos que no son de movimiento.

Ejemplos:
Tantata mikuni ‘yo como pan’
Por el contrario con los verbos de movimiento –ta indica hacia, que es igual a meta.

Ejemplo:
Ayacuchuta rini ‘voy hacia Ayacucho’

&gt; Caso genitivo:
Su marca es –pa, que es igual a la preposición “de”.

Ejemplos:
Runapa ‘de la gente’
Allpapa ‘de la tierra’

&g

## Step 4: Generar traducción usando contexto combinado

In [12]:
translation_prompt = f"""
Answer the question based on the context from both documents:

Question: {translation_query}

Context: {combined_content}

Provide clear, accurate translations for each of the 5 Quechua phrases based only on the context provided.
Include explanations of the grammatical structures used when relevant.
"""

print('Generated Translation:\n')
translation_answer = await gemini_client.generate_text(translation_prompt)
utils.print_wrapped(translation_answer)

# Guardar la traducción en un archivo de texto
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
translation_output_file = root_dir / "data" / f"quechua_translation_{timestamp}.txt"

with open(translation_output_file, 'w', encoding='utf-8') as f:
    f.write("Translation Query:\n")
    f.write(translation_query)
    f.write("\n" + "="*80 + "\n\n")
    f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    f.write("Translation Result:\n")
    f.write(translation_answer)
    f.write("\n\n" + "="*80 + "\n")
    f.write(f"Sources Used:\n")
    f.write(f"\nFrom Quechua Grammar ({len(quechua_nodes_list)} nodes):\n")
    for node_id in quechua_nodes_list:
        node = quechua_node_map[node_id]
        f.write(f"- Node {node['node_id']} (Page {node['page_index']}): {node['title']}\n")
    f.write(f"\nFrom Gramar ({len(gramar_nodes_list)} nodes):\n")
    for node_id in gramar_nodes_list:
        node = gramar_node_map[node_id]
        f.write(f"- Node {node['node_id']} (Page {node['page_index']}): {node['title']}\n")

print(f'\nTranslation saved to: {translation_output_file}')

Generated Translation:

Basado en los documentos de gramática quechua proporcionados, aquí tienes las traducciones de las 5
frases solicitadas, junto con la explicación de los elementos utilizados según el contexto:

### 1. Ñuqa hatun wasipi tiyani
**Traducción:** **Yo vivo en la casa grande.**

*   **Ñuqa:** Pronombre personal de primera persona singular "Yo" (Doc 2, § 3).
*   **Hatun:** Adjetivo que significa "grande" (Doc 1, 3.1.4).
*   **Wasi-pi:** Sustantivo "casa" + sufijo locativo **-pi** que significa "en" (Doc 1, Caso
locativo; Doc 2, § 6.6).
*   **Tiya-ni:** Verbo "vivir" + sufijo de primera persona **-ni** (Doc 2, § 2.1).

### 2. Qam imaynataq kasanki
**Traducción:** **¿Cómo estás (tú)?**

*   **Qam:** Pronombre personal de segunda persona singular "Tú" (Doc 2, § 3).
*   **Imayna-taq:** Pronombre interrogativo "Cómo" (Doc 1, 2.1.5; Doc 2, § 7) + sufijo **-taq**
utilizado para formular preguntas o contrastes (Doc 1, 1.3.2.2; Doc 2, § 7).
*   **Kasanki:** Del verbo *kay* ("ser